# Day 1 - EDA, Alignment Validation, and Persistence Baseline

This notebook is designed for the Kaggle competition environment. It loads `train_raw.csv`, `test.csv`, and `sample_submission.csv`, validates one-hour-ahead alignment on raw training data, evaluates a persistence baseline locally, and creates `submission_persistence.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

COMPETITION_SLUG = 'co-5420-air-pollution-forecasting-using-temporal-n-ns'
TARGET = 'PM2.5'
STATION_COL = 'station'
DATETIME_COL = 'datetime'
RAW_TIME_COLS = ['year', 'month', 'day', 'hour']
RAW_NUMERIC_FEATURES = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

def find_data_dir():
    candidates = [
        Path('/kaggle/input') / COMPETITION_SLUG,
        Path('/kaggle/input'),
        Path('../input') / COMPETITION_SLUG,
        Path('../input'),
        Path('data/raw'),
    ]
    for candidate in candidates:
        if (candidate / 'train_raw.csv').exists() or (candidate / 'test.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find Kaggle CSV files.')

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
REPORT_DIR = OUTPUT_DIR / 'reports' / 'day1'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR, OUTPUT_DIR

In [ ]:
train_raw = pd.read_csv(DATA_DIR / 'train_raw.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(train_raw.shape, test.shape, sample_submission.shape)
display(train_raw.head())
display(test.head())

In [ ]:
eda_summary = pd.DataFrame({
    'metric': ['rows', 'columns', 'stations', 'year_min', 'year_max', 'pm25_missing', 'pm25_mean', 'pm25_median', 'pm25_std'],
    'value': [
        len(train_raw), len(train_raw.columns), train_raw['station'].nunique(),
        train_raw['year'].min(), train_raw['year'].max(), train_raw['PM2.5'].isna().sum(),
        train_raw['PM2.5'].mean(), train_raw['PM2.5'].median(), train_raw['PM2.5'].std(),
    ]
})

missing_values = (
    train_raw.isna().sum().rename('missing_count').to_frame()
    .assign(missing_fraction=lambda x: x['missing_count'] / len(train_raw))
    .reset_index(names='column')
    .sort_values('missing_count', ascending=False)
)

eda_summary.to_csv(REPORT_DIR / 'eda_summary.csv', index=False)
missing_values.to_csv(REPORT_DIR / 'missing_values.csv', index=False)
display(eda_summary)
display(missing_values.head(20))

In [ ]:
def add_datetime(df):
    out = df.copy()
    out[DATETIME_COL] = pd.to_datetime(out[RAW_TIME_COLS])
    return out

def sort_raw(df):
    out = add_datetime(df) if DATETIME_COL not in df.columns else df.copy()
    return out.sort_values([STATION_COL, DATETIME_COL]).reset_index(drop=True)

def stationwise_time_impute(df):
    out = sort_raw(df)
    numeric_cols = [col for col in RAW_NUMERIC_FEATURES if col in out.columns]
    out[numeric_cols] = out.groupby(STATION_COL, group_keys=False)[numeric_cols].ffill()
    out[numeric_cols] = out.groupby(STATION_COL, group_keys=False)[numeric_cols].bfill()
    out[numeric_cols] = out[numeric_cols].fillna(out[numeric_cols].median(numeric_only=True))
    if 'wd' in out.columns:
        out['wd'] = out.groupby(STATION_COL, group_keys=False)['wd'].ffill()
        out['wd'] = out.groupby(STATION_COL, group_keys=False)['wd'].bfill()
        mode = out['wd'].mode(dropna=True)
        out['wd'] = out['wd'].fillna(mode.iloc[0] if len(mode) else 'UNKNOWN')
    return out

def make_one_hour_ahead_frame(df):
    out = sort_raw(df)
    out['target_next_pm25'] = out.groupby(STATION_COL)[TARGET].shift(-1)
    out['next_datetime'] = out.groupby(STATION_COL)[DATETIME_COL].shift(-1)
    out['hours_to_target'] = (out['next_datetime'] - out[DATETIME_COL]).dt.total_seconds() / 3600.0
    return out[(out['hours_to_target'] == 1.0) & out['target_next_pm25'].notna()].copy().reset_index(drop=True)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float))))

In [ ]:
sorted_train = sort_raw(train_raw)
candidate_rows = len(sorted_train) - sorted_train[STATION_COL].nunique()
aligned_raw = make_one_hour_ahead_frame(sorted_train)
alignment_validation = pd.DataFrame([{
    'raw_rows': len(sorted_train),
    'stations': sorted_train[STATION_COL].nunique(),
    'candidate_one_step_rows': candidate_rows,
    'valid_one_hour_rows': len(aligned_raw),
    'dropped_nonconsecutive_or_missing_target_rows': candidate_rows - len(aligned_raw),
}])
alignment_validation.to_csv(REPORT_DIR / 'alignment_validation.csv', index=False)
display(alignment_validation)

In [ ]:
imputed_train = stationwise_time_impute(train_raw)
aligned = make_one_hour_ahead_frame(imputed_train).sort_values(DATETIME_COL).reset_index(drop=True)

split_idx = int(len(aligned) * 0.8)
valid = aligned.iloc[split_idx:].copy()
y_true = valid['target_next_pm25'].astype(float)
y_pred = valid[TARGET].astype(float)

metrics = pd.DataFrame([{
    'model': 'persistence_latest_pm25',
    'validation_rows': len(valid),
    'rmse': rmse(y_true, y_pred),
    'mae': mae(y_true, y_pred),
    'validation_fraction': 0.2,
}])
metrics.to_csv(REPORT_DIR / 'persistence_validation_metrics.csv', index=False)
display(metrics)

In [ ]:
def persistence_from_test(test_df):
    for col in ['PM2.5_lag_1', 'PM2.5_lag_01']:
        if col in test_df.columns:
            return test_df[col].astype(float)
    lag_cols = [col for col in test_df.columns if col.startswith('PM2.5') and col.endswith('_lag_1')]
    if not lag_cols:
        raise ValueError('Could not find PM2.5_lag_1 in test.csv')
    return test_df[lag_cols[0]].astype(float)

pred = persistence_from_test(test)
submission = sample_submission[['id']].copy()
submission['PM2.5'] = pred.fillna(pred.median())
submission_path = SUBMISSION_DIR / 'submission_persistence.csv'
submission.to_csv(submission_path, index=False)
print(submission_path)
display(submission.head())
display(submission.tail())
print(submission.shape)